## 🔧 3. Agents with Built in tools

**Goal**: Enhance agents with built-in tools — **Code Interpreter**, **File Search**, **Web Search**, and custom **Function Tools** — registered server-side in Foundry with full **tracing**, then used via `FoundryAgent`.

- `CodeInterpreterTool` — data analysis & charts (server-side)
- `FileSearchTool` — RAG over uploaded documents (server-side)
- `WebSearchTool` — real-time web search (server-side)
- `FunctionTool` + `@tool` decorator — custom Python functions
- **Tracing** — all interactions visible in Foundry portal

In [6]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, os
from pathlib import Path
sys.path.insert(0, os.path.abspath("..") if "notebooks" in os.getcwd() else ".")

from dotenv import load_dotenv
load_dotenv()

from utils.config import get_env, get_credential
from agent_framework import tool

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
    CodeInterpreterTool,
    FileSearchTool,
    WebSearchTool,
    FunctionTool as AzureFunctionTool,
)
from agent_framework.foundry import FoundryAgent

# Endpoints
PROJECT_ENDPOINT = get_env("AZURE_AI_PROJECT_ENDPOINT_FULL")
MODEL = get_env("AZURE_AI_MODEL_DEPLOYMENT_NAME")

project = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=get_credential(),
)

print(f"📡 Project endpoint: {PROJECT_ENDPOINT}")
print(f"🧠 Model: {MODEL}")
print("✅ Connected to Foundry")

📡 Project endpoint: https://foundry-demo-us-east-resource.services.ai.azure.com/api/projects/prj-foundry-demo-us-east-dev-001
🧠 Model: gpt-4o
✅ Connected to Foundry


---
### Part A — Code Interpreter 🧮
Register an agent with `CodeInterpreterTool` in Foundry, then use it via `FoundryAgent`.
The agent writes and executes Python code to analyze data and create charts — all traced server-side.

In [7]:
# ── A1. Register Code Interpreter agent in Foundry ───────────────────────────
CODE_AGENT_NAME = "DataAnalyst-af"

code_agent_version = project.agents.create_version(
    agent_name=CODE_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a data analyst. When given data, write and execute Python "
            "code to analyze it. Always produce a chart when asked for visualization. "
            "Explain your findings clearly."
        ),
        tools=[CodeInterpreterTool()],
    ),
)

# Connect via Agent Framework
code_agent = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT,
    agent_name=code_agent_version.name,
    agent_version=str(code_agent_version.version),
    credential=get_credential(),
)

print(f"✅ Agent registered in Foundry: {code_agent_version.name} v{code_agent_version.version}")
print(f"🤖 FoundryAgent connected: {type(code_agent).__name__}")

✅ Agent registered in Foundry: DataAnalyst-af v1
🤖 FoundryAgent connected: FoundryAgent


In [8]:
# ── A2. Ask for analysis with code execution ─────────────────────────────────
from agent_framework import Content

result = await code_agent.run(
    "Calculate the first 20 Fibonacci numbers and create a line chart showing their growth."
)
print(f"🗨️  Agent:\n{result}")

# Inspect generated code
for message in result.messages:
    code_blocks = [c for c in message.contents if c.type == "code_interpreter_tool_call"]
    outputs = [c for c in message.contents if c.type == "code_interpreter_tool_result"]
    if code_blocks:
        for content in (code_blocks[0].inputs or []):
            if isinstance(content, Content) and content.type == "text":
                print(f"\n📝 Generated code:\n{content.text}")
                break
    if outputs:
        for out in (outputs[0].outputs or []):
            if isinstance(out, Content) and out.type == "text":
                print(f"\n📊 Execution output:\n{out.text}")

print(f"\n🔍 Check Foundry portal → Tracing to see this interaction!")

🗨️  Agent:
The line chart above depicts the first 20 Fibonacci numbers, showcasing their exponential growth. The Fibonacci sequence starts with 0 and 1, and each subsequent number is the sum of the two preceding ones. 

Initially, the growth is slow, but as the sequence progresses, the numbers increase rapidly, highlighting the characteristic exponential pattern of the Fibonacci sequence. This serves as a visual demonstration of how quickly Fibonacci numbers can grow.

📝 Generated code:
import matplotlib.pyplot as plt

# Function to calculate Fibonacci numbers
def fibonacci(n):
    fib_sequence = [0, 1]
    while len(fib_sequence) < n:
        fib_sequence.append(fib_sequence[-1] + fib_sequence[-2])
    return fib_sequence

# Calculate the first 20 Fibonacci numbers
fib_numbers = fibonacci(20)

# Plot the Fibonacci numbers
plt.figure(figsize=(10, 6))
plt.plot(range(1, 21), fib_numbers, marker='o', linestyle='-')
plt.title('First 20 Fibonacci Numbers')
plt.xlabel('Position in Sequence')

---
### Part B — Web Search 🌐
Register an agent with `WebSearchTool` in Foundry to fetch real-time info from the web, all traced server-side.

In [9]:
# ── C1. Register Web Search agent in Foundry ─────────────────────────────────
WEB_AGENT_NAME = "ResearchAssistant-af"

web_agent_version = project.agents.create_version(
    agent_name=WEB_AGENT_NAME,
    definition=PromptAgentDefinition(
        model=MODEL,
        instructions=(
            "You are a research assistant. Use web search to find the latest "
            "information and always include source URLs in your response."
        ),
        tools=[WebSearchTool()],
    ),
)

# Connect via Agent Framework
web_agent = FoundryAgent(
    project_endpoint=PROJECT_ENDPOINT,
    agent_name=web_agent_version.name,
    agent_version=str(web_agent_version.version),
    credential=get_credential(),
)

print(f"✅ Agent registered in Foundry: {web_agent_version.name} v{web_agent_version.version}")
print(f"🤖 FoundryAgent connected: {type(web_agent).__name__}")

result = await web_agent.run(
    "What are the top AI announcements from Microsoft Build this year?"
)
print(f"\n🗨️  Agent: {result}")
print(f"\n🔍 Check Foundry portal → Tracing to see this interaction!")

✅ Agent registered in Foundry: ResearchAssistant-af v1
🤖 FoundryAgent connected: FoundryAgent

🗨️  Agent: Here’s a structured, in-depth analysis (≈850 words) of the top AI announcements from Microsoft Build 2026, held June 2–3. All key insights are cited using multiple sources, and URLs are included as requested.

---

##  Key AI Announcements at Microsoft Build 2026

### 1. The MAI Model Family – Seven New In‑House AI Models

- Microsoft unveiled the **MAI (Microsoft AI)** model family—seven new models spanning reasoning, vision, transcription, voice, and coding. This marks a strategic move toward building and controlling its own model layer rather than relying solely on external providers ([microsoft.ai](https://microsoft.ai/news/microsoft-build-2026-mai-keynote-transcript/)).

- **MAI‑Thinking‑1**: A reasoning model with a large 256,000‑token context window and 35 billion parameters. Designed for complex reasoning tasks and enterprise use cases where compliance and long-context hand

---
### Cleanup 🧹

In [10]:
# ── Cleanup ───────────────────────────────────────────────────────────────────
# Delete agents from Foundry
for name in [CODE_AGENT_NAME, WEB_AGENT_NAME]:
    try:
        project.agents.delete(agent_name=name)
        print(f"🧹 Deleted agent: {name}")
    except Exception as e:
        print(f"⚠️  Could not delete {name}: {e}")

# Delete vector store and file
try:
    oai.vector_stores.delete(vector_store.id)
    oai.files.delete(file.id)
    print("🧹 Vector store and file deleted")
except Exception:
    pass

print("✅ Cleanup complete")

🧹 Deleted agent: DataAnalyst-af
🧹 Deleted agent: ResearchAssistant-af
✅ Cleanup complete


---
### ✅ What you demonstrated
| Capability | Foundry Definition | Agent Framework Feature |
|---|---|---|
| Data analysis & charts | `CodeInterpreterTool()` on `PromptAgentDefinition` | Runs server-side via `FoundryAgent` |
| Document Q&A (RAG) | `FileSearchTool(vector_store_ids=[...])` | Runs server-side via `FoundryAgent` |
| Real-time web knowledge | `WebSearchTool()` | Runs server-side via `FoundryAgent` |
| Server-side tracing | All interactions visible in Foundry portal → Tracing | |

**Next →** [04_agent_tools_mcp.ipynb](./04_agent_tools_mcp.ipynb) — Agent with MCP tools.